# Sketch → Image: full experiment run (Kaggle GPU)

This notebook reproduces **all results** of the project: it installs the dependencies, runs every
experiment defined in `configs/experiments.yaml`, computes the metrics, draws the comparison grids and
packs everything into `results.zip`.

**Before running** (see `docs/KAGGLE_GUIDE.md`):
1. *Settings → Accelerator → GPU T4 x2* (or a single T4)
2. *Settings → Internet → On* (models are downloaded from Hugging Face)
3. Set `REPO_URL` below, **or** attach the project as a Kaggle Dataset

Then click **Save Version → Save & Run All (Commit)**. It runs in the background (≈ 60–75 min);
the outputs appear in the *Output* tab of the saved version.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4"

## 1. Get the code
From GitHub, or from a Kaggle Dataset containing the project folder (auto-detected).

In [ ]:
import glob, os, shutil, subprocess

REPO_URL = "https://github.com/Saddam7342/sketch-to-image-genai.git"
WORK = "/kaggle/working/sketch2image"

found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
if not os.path.isdir(WORK):
    if found:
        print("Using Kaggle Dataset copy:", found[0])
        shutil.copytree(found[0], WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
print(sorted(os.listdir()))

## 2. Install dependencies
Kaggle already has a CUDA build of PyTorch; we only add the diffusion stack.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q controlnet-aux==0.0.10 || echo "controlnet-aux failed: experiment E1b will be skipped"
!pip install -q -e . --no-deps
import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)

## 3. Sanity check: one image
If this works, the full sweep will work too.

In [ ]:
!python scripts/generate.py --sketch sketches/02_robot_character.png \
    --subject "a friendly small robot character waving one hand, round helmet head with a glowing visor" \
    --style photoreal --control scribble --seeds 1 --out outputs/sanity
from IPython.display import Image, display
display(Image("outputs/sanity/02_robot_character__photoreal__scribble__seed1.png", width=384))

## 4. Main experiments (SD 1.5 + ControlNet v1.1)
Images that already exist are skipped, so you can simply re-run this cell after an interruption.

In [ ]:
!python scripts/run_experiments.py --config configs/experiments.yaml --out results \
    --only E1_control_type E2_cond_scale E3_control_window E4_style E5_cfg_steps E6_lcm E7_seeds E8_preprocess

## 5. Learned annotators (optional, needs controlnet-aux)

In [ ]:
try:
    import controlnet_aux  # noqa: F401
    !python scripts/run_experiments.py --config configs/experiments.yaml --out results --only E1b_learned_annotators
except ImportError:
    print("controlnet-aux not installed: skipping E1b")

## 6. SDXL comparison (1024 px, larger download ≈ 7 GB)

In [ ]:
!python scripts/run_experiments.py --config configs/experiments.yaml --out results --only E9_sdxl

## 7. Analysis: plots + summary tables

In [ ]:
!python scripts/analyze.py --config configs/experiments.yaml --results results
from IPython.display import Markdown
display(Markdown(open("results/analysis/summary.md", encoding="utf-8").read()))

## 8. Comparison grids

In [ ]:
from IPython.display import Image, display
for g in sorted(glob.glob("results/*/grid.jpg")):
    print(g)
    display(Image(g, width=1100))

## 9. Pack the results
Download `results.zip` from the *Output* tab (or the folder icon on the right).

In [ ]:
!cd /kaggle/working && zip -qr results.zip sketch2image/results && ls -lh /kaggle/working/results.zip